# Simple Neural Network With PyTorch

In this lab, we'll build several neural networks: a simple one first (without a hidden layer), then a multilayer perceptron (several hidden layers). We'll use the MNIST dataset to test how this network learns in real conditions.

MNIST is the *Hello World!* dataset of machine learning. It's made of 28 $\times$ 28 pixel grayscale images. These images represent digits (0 to 9). Each image comes with a label giving the character the image is supposed to represent.

## Importing the Libraries

PyTorch and the MNIST dataset

In [ ]:
import collections.abc
import random

import numpy
import matplotlib.pyplot as plt
import seaborn
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Computations run on the GPU if there is one
device = "cuda" if torch.cuda.is_available() else "cpu"

[`matplotlib`](https://matplotlib.org/), [`numpy`](https://numpy.org/) and [`seaborn`](https://seaborn.pydata.org/) are basic libraries for machine learning with Python, which we'll use here for visualization and simple matrix operations

## Getting Started With Tensors

On Colab, PyTorch is already installed. On your machine, install it with `pip install torch torchvision`, or with the command suited to your system and GPU given by [pytorch.org](https://pytorch.org/get-started/locally/).

Before building networks, let's handle PyTorch tensors, whose API is very close to NumPy's (see the "NumPy and PyTorch" and "Computing on GPU" slides).

1. *Create a tensor `a` of shape (2, 3) holding the integers from 0 to 5 ([`torch.arange`](https://docs.pytorch.org/docs/stable/generated/torch.arange.html), then `reshape`). What is its `dtype`? Convert it to floats (`.float()`), then create a tensor `b` of shape (3,) filled with 1s ([`torch.ones`](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)).*
2. *Compute `a + b` and the matrix product `a @ b`. What are the shapes of the results? Why does `a + torch.ones(2)` fail?*
3. *Extract the second column of `a`, then its last row. Add a dimension of size 1 in front of `a` with `unsqueeze(0)`: what shape do you get? What is such a dimension for with a PyTorch model?*
4. *Convert `a` to a NumPy array, then back to a tensor. Copy `a` to `device` (defined in the imports cell) and display its `device` attribute.*

In [ ]:
# Your code here

### Solution

In [ ]:
a = torch.arange(6).reshape(2, 3)
print(a, a.dtype)
a = a.float()
b = torch.ones(3)

# b is broadcast over each row of a
print(a + b, (a + b).shape)
# Matrix-vector product: the sum of each row of a
print(a @ b, (a @ b).shape)

print(a[:, 1])               # second column
print(a[-1])                 # last row
print(a.unsqueeze(0).shape)  # batch dimension of size 1

a_numpy = a.numpy()
print(type(a_numpy), torch.from_numpy(a_numpy).dtype)
print(a.to(device).device)

- `torch.arange` creates 64-bit integers (`torch.int64`); neural networks compute with floats, `torch.float32` by default.
- `a + b` has shape (2, 3) and `a @ b` shape (2,). For broadcasting, dimensions are compared starting from the last one: they must be equal, or one of them must be 1 or missing. (2, 3) and (3,) are compatible, but not (2, 3) and (2,): it would take the shape (2, 1), `torch.ones(2).unsqueeze(1)`.
- `a.unsqueeze(0)` has shape (1, 2, 3). PyTorch models process batches, the first dimension being the examples': to predict on a single example, we add a batch dimension of size 1 to it.
- `a.numpy()` shares the tensor's memory, without a copy: this requires a tensor on CPU (`.cpu()`) and outside the autograd graph (`.detach()`). `torch.from_numpy` keeps the NumPy type (`float32` here, but `float64` for a NumPy array created by default).

## Fetching the Data

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
X_train, y_train = train_data.data, train_data.targets
X_test, y_test = test_data.data, test_data.targets

## Let's Look at the Data

Use the next cell to explore the `X_train`, `y_train`, `X_test` and `y_test` tensors. Feel free to use autocompletion to browse their attributes and methods.

To do machine learning, we look for training and test sets, and display a few examples with their associated labels.

Try to answer the following questions:
- How many images are there in total in this dataset?
- In what form are the images stored? The labels?
- Are the classes balanced?

In [ ]:
# Your code here

### Solution

In [ ]:
example = X_train[0]
example_label = int(y_train[0])

print(f"Examples shape: {tuple(X_train.shape)}")
print(f"Labels shape: {tuple(y_train.shape)}")

plt.imshow(example, cmap="gray_r")
plt.title(f"First example of the dataset ({example_label})")
plt.show()

seaborn.countplot(x=y_train.numpy())
plt.title("Counts of the different classes (digits)")
plt.ylabel("Count")
plt.xlabel("Digit")
plt.show()

## Displaying Examples

Display 25 examples, drawn at random from the training set. Don't forget to display the corresponding label in an easy-to-read form.

To do so, use:
- [`numpy.random.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html)
- [`matplotlib.pyplot.imshow`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.imshow.html)
- [`matplotlib.pyplot.subplots`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.subplots.html) for an elegant display

In [ ]:
# Use subplots for a large 5x5 grid display
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# Your code here

# ax[2, 4].imshow(... , cmap="gray_r")
# ax[2, 4].set_title("Example n (label)")

### Solution

In [ ]:
# Create the grid of subplots. We pass the figsize argument to enlarge
# the figure, which is small by default
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# Pick 25 random indices, without replacement (we don't want to display the
# same image twice)
random_indexes = numpy.random.choice(X_train.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = X_train[img_index]
    label = int(y_train[img_index])

    # Display with matplotlib and its imshow function, very handy in computer
    # vision
    ax[i, j].imshow(image, cmap='gray_r')
    ax[i, j].set_title(f"Example {img_index} ({label})")
    ax[i, j].axis('off')

## Transforming the Data

We need to apply a few transformations to this data:

1. We could keep the original tensor shapes `(_, 28, 28)`, but it will be easier to work on tensors of shape `(_, 28²)`, where `_` is the original number of examples.
2. We want to use the $[0, 1]$ range rather than values between $0$ and $255$ for our images. Transform the inputs accordingly.

Useful function: [`torch.Tensor.reshape`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)

*Apply the first two transformations to `X_train` and `X_test`.*

In [ ]:
# Your code here

### Solution

In [ ]:
nb_classes = 10
input_dim = 28 * 28

# We want to "flatten" X so that our network's input is a vector
# of size input_dim
X_train = X_train.reshape(X_train.shape[0], input_dim)
# equivalent syntax
X_train = X_train.reshape(-1, input_dim)

X_test = X_test.reshape(X_test.shape[0], input_dim)
# equivalent syntax
X_test = X_test.reshape(-1, input_dim)

# Transform to [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

# Move the tensors to the compute device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

## Train / Validation Split

To choose the hyperparameters (learning rate, network size…), we compare models on a **validation** set. The test set is reserved for the final evaluation: using it to make choices would make that evaluation too optimistic.

We set aside the last 10,000 training examples for validation:

In [ ]:
X_train, X_val = X_train[:50000], X_train[50000:]
y_train, y_val = y_train[:50000], y_train[50000:]
print(X_train.shape, X_val.shape, X_test.shape)

## Building a Neural Network Without a Hidden Layer

### Model Variables

In a neural network without a hidden layer, $y = \sigma(XW +b)$, where $\sigma$ is a function suited to the problem. Here, we'll choose the softmax function, since we're tackling a multi-class classification problem.

As in the course, `W` denotes the weight matrix, multiplied by the input, and `b` the bias vector, added afterwards.

*Complete the `create_simple_nn_weights` function to initialize `W` and `b` as [PyTorch parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) with the [`torch.randn`](https://docs.pytorch.org/docs/stable/generated/torch.randn.html) function.*

In [ ]:
def create_simple_nn_weights(n_inputs: int,
                             n_outputs: int
                             ) -> tuple[nn.Parameter, nn.Parameter]:
  W = None  # Your code here
  b = None  # Your code here
  return W, b

#### Solution

In [ ]:
def create_simple_nn_weights(n_inputs: int,
                             n_outputs: int
                             ) -> tuple[nn.Parameter, nn.Parameter]:
  return (nn.Parameter(torch.randn(n_inputs, n_outputs, device=device)),
          nn.Parameter(torch.randn(n_outputs, device=device)))

### Defining the Model

Here, we'll prepare our model's architecture.

*Fix the following code so that `y_pred` is the result of our neural network without hidden layers: $\text{softmax}(XW+b)$. We'll use the [log of the softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html) rather than the standard softmax for better numerical stability.*

In [ ]:
class SimpleNN(nn.Module):
  def __init__(self, n_inputs: int = 28 ** 2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    y_pred = X  # Your code here
    return y_pred


simple_nn = SimpleNN()
simple_nn(X_train)

#### Solution

In [ ]:
# Create the model
class SimpleNN(nn.Module):
  def __init__(self, n_inputs: int = 28 ** 2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    return torch.log_softmax(X @ self.W + self.b, dim=-1)


simple_nn = SimpleNN()
simple_nn(X_train)

### Computing the Loss Function

We'll use a standard loss function for multi-class classification: cross-entropy. Since our model already returns log-probabilities, you can use [`torch.nn.functional.nll_loss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.nll_loss.html). Moreover, we want to be able to interpret this value directly, so the `categorical_crossentropy` function must return an aggregated value (the mean): call `nll_loss` with `reduction="none"`, which returns one value per example.

You can use [`torch.mean`](https://docs.pytorch.org/docs/stable/generated/torch.mean.html) for that.

*Complete the `categorical_crossentropy` function.*

In [ ]:
def categorical_crossentropy(y: torch.Tensor,
                             y_pred: torch.Tensor) -> torch.Tensor:
  loss = None  # Your code here
  return loss


simple_nn = SimpleNN()
y_train_pred = simple_nn(X_train)
categorical_crossentropy(y_train, y_train_pred)

#### Solution

In [ ]:
def categorical_crossentropy(y: torch.Tensor,
                             y_pred: torch.Tensor) -> torch.Tensor:
  cross_entropy = nn.functional.nll_loss(y_pred, y, reduction="none")
  return torch.mean(cross_entropy)


simple_nn = SimpleNN()
y_train_pred = simple_nn(X_train)
categorical_crossentropy(y_train, y_train_pred)

### Metrics

To know whether a model learns properly, it's important to measure its performance. In this lab, we'll use the simplest but also one of the most informative performance measures: accuracy. It's simply the number of correct predictions over the total number of predictions.

*Use [torch.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html) and tensor comparisons to complete the accuracy function.*

In [ ]:
def accuracy(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  predictions = None  # Your code here
  return None


simple_nn = SimpleNN()
y_test_pred = simple_nn(X_test)
accuracy(y_test, y_test_pred)

#### Solution

In [ ]:
def accuracy(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  argmaxed = torch.argmax(y_pred, dim=-1)
  n_equal = torch.count_nonzero(argmaxed == y)
  return (n_equal / y.shape[0]).item()


simple_nn = SimpleNN()
y_test_pred = simple_nn(X_test)
accuracy(y_test, y_test_pred)

## Creating a PyTorch `DataLoader`

- *Use [`torch.utils.data.TensorDataset`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset) to write the `create_dataset` function, which creates a PyTorch dataset from the `X_train` and `y_train` tensors. We want batches of the form `(X_batch, y_batch)`.*
- *Wrap this dataset in a [`DataLoader`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) to group the examples into batches.*
- *Use the `shuffle` argument so that the dataset is shuffled at each epoch.*

In [ ]:
def create_dataset(X: torch.Tensor, y: torch.Tensor, batch_size: int = 4000
                   ) -> DataLoader:
  pass  # Your code here

### Solution

In [ ]:
def create_dataset(X: torch.Tensor, y: torch.Tensor, batch_size: int = 4000
                   ) -> DataLoader:
  dataset = TensorDataset(X, y)
  dataset = DataLoader(dataset, batch_size=batch_size, shuffle=True)
  return dataset

## Training

Here's a training loop that's almost set up.

*Implement the missing part, which updates the parameters for a given batch. To do so, you'll use [PyTorch's automatic differentiation](https://docs.pytorch.org/docs/stable/notes/autograd.html): `loss.backward()` computes the gradient of the loss and stores it in the `.grad` attribute of each parameter.*

In [ ]:
LossFunction = collections.abc.Callable[[torch.Tensor, torch.Tensor],
                                       torch.Tensor]
RegularizationFunction = collections.abc.Callable[[nn.Module], torch.Tensor]


def train(model: nn.Module,
          X_train: torch.Tensor = X_train,
          y_train: torch.Tensor = y_train,
          X_val: torch.Tensor = X_val,
          y_val: torch.Tensor = y_val,
          epochs: int = 150,
          batch_size: int = 4000,
          learning_rate: float = 0.001,
          evaluate_every: int = 10,
          loss: LossFunction = categorical_crossentropy,
          regularization: RegularizationFunction | None = None,
         ) -> tuple[list[float], ...]:
  with torch.no_grad():
    y_train_pred = model(X_train)
    y_val_pred = model(X_val)
    accuracies = [accuracy(y_train, y_train_pred)]
    val_accuracies = [accuracy(y_val, y_val_pred)]
    losses = [float(loss(y_train, y_train_pred))]
    val_losses = [float(loss(y_val, y_val_pred))]

  print(f"Initial metrics: acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}")

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for e in range(epochs):

    for X_train_batch, y_train_batch in dataset_train:

      pass # Your code here

    # Compute and display the evaluation metric on the validation set
    # every `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      with torch.no_grad():
        y_train_pred = model(X_train)
        y_val_pred = model(X_val)
        accuracies.append(accuracy(y_train, y_train_pred))
        losses.append(float(loss(y_train, y_train_pred)))
        val_accuracies.append(accuracy(y_val, y_val_pred))
        val_losses.append(float(loss(y_val, y_val_pred)))
      print(f"Epoch {e + 1} metrics: acc {accuracies[-1]:.4f}, "
            f"val_acc {val_accuracies[-1]:.4f}, "
            f"loss {losses[-1]:.4f}, "
            f"val_loss {val_losses[-1]:.4f}")

  x_ticks = numpy.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Training")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Loss function during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Training")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Accuracy during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses


simple_nn = SimpleNN()
_ = train(simple_nn, epochs=100, learning_rate=3)

### Solution

In [ ]:
LossFunction = collections.abc.Callable[[torch.Tensor, torch.Tensor],
                                       torch.Tensor]
RegularizationFunction = collections.abc.Callable[[nn.Module], torch.Tensor]


def train(model: nn.Module,
          X_train: torch.Tensor = X_train,
          y_train: torch.Tensor = y_train,
          X_val: torch.Tensor = X_val,
          y_val: torch.Tensor = y_val,
          epochs: int = 150,
          batch_size: int = 4000,
          learning_rate: float = 0.001,
          evaluate_every: int = 10,
          loss: LossFunction = categorical_crossentropy,
          regularization: RegularizationFunction | None = None,
         ) -> tuple[list[float], ...]:
  with torch.no_grad():
    y_train_pred = model(X_train)
    y_val_pred = model(X_val)
    accuracies = [accuracy(y_train, y_train_pred)]
    val_accuracies = [accuracy(y_val, y_val_pred)]
    losses = [float(loss(y_train, y_train_pred))]
    val_losses = [float(loss(y_val, y_val_pred))]

  print(f"Initial metrics: acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}")

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for e in range(epochs):

    for X_train_batch, y_train_batch in dataset_train:

      # Compute the loss
      y_train_batch_pred = model(X_train_batch)
      loss_scalar = loss(y_train_batch, y_train_batch_pred)
      if regularization is not None:
        loss_scalar = loss_scalar + regularization(model)

      # Automatically compute the gradient of the loss with respect to each
      # parameter: it's stored in the parameter's .grad attribute
      model.zero_grad()
      loss_scalar.backward()

      # Go through the parameters one by one to update them following the
      # rule: new_value = old_value - learning_rate * gradient
      with torch.no_grad():
        for parameter in model.parameters():
          parameter -= parameter.grad * learning_rate

    # Compute and display the evaluation metric on the validation set
    # every `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      with torch.no_grad():
        y_train_pred = model(X_train)
        y_val_pred = model(X_val)
        accuracies.append(accuracy(y_train, y_train_pred))
        losses.append(float(loss(y_train, y_train_pred)))
        val_accuracies.append(accuracy(y_val, y_val_pred))
        val_losses.append(float(loss(y_val, y_val_pred)))
      print(f"Epoch {e + 1} metrics: acc {accuracies[-1]:.4f}, "
            f"val_acc {val_accuracies[-1]:.4f}, "
            f"loss {losses[-1]:.4f}, "
            f"val_loss {val_losses[-1]:.4f}")

  x_ticks = numpy.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Training")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Loss function during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Training")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Accuracy during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses


simple_nn = SimpleNN()
_ = train(simple_nn, epochs=100, learning_rate=3)

## With a PyTorch Optimizer

The update written by hand above (`parameter -= learning_rate * gradient`) is exactly what the [`torch.optim.SGD`](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html) optimizer does. In practice, we therefore write the standard training loop (see the "Complete Training Loop" slide):

```python
optimizer.zero_grad()  # reset the gradients of the previous batch
loss.backward()        # compute the gradients
optimizer.step()       # update the parameters
```

*Complete the loop of `train_with_optimizer`, train a `SimpleNN` with `torch.optim.SGD` and the same learning rate as before: do you get the same performance? Then try [`torch.optim.Adam`](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html), with a much smaller learning rate (for instance 0.01).*

In [ ]:
def train_with_optimizer(model: nn.Module,
                         optimizer: torch.optim.Optimizer,
                         epochs: int = 100,
                         batch_size: int = 4000,
                         evaluate_every: int = 10) -> None:
  dataset_train = create_dataset(X_train, y_train, batch_size)
  for e in range(epochs):
    for X_train_batch, y_train_batch in dataset_train:
      pass  # Your code here
    if (e + 1) % evaluate_every == 0:
      with torch.no_grad():
        val_accuracy = accuracy(y_val, model(X_val))
      print(f"Epoch {e + 1}: val_acc {val_accuracy:.4f}")


simple_nn = SimpleNN()
train_with_optimizer(simple_nn, torch.optim.SGD(simple_nn.parameters(), lr=3))

### Solution

In [ ]:
def train_with_optimizer(model: nn.Module,
                         optimizer: torch.optim.Optimizer,
                         epochs: int = 100,
                         batch_size: int = 4000,
                         evaluate_every: int = 10) -> None:
  dataset_train = create_dataset(X_train, y_train, batch_size)
  for e in range(epochs):
    for X_train_batch, y_train_batch in dataset_train:
      optimizer.zero_grad()
      loss = categorical_crossentropy(y_train_batch, model(X_train_batch))
      loss.backward()
      optimizer.step()
    if (e + 1) % evaluate_every == 0:
      with torch.no_grad():
        val_accuracy = accuracy(y_val, model(X_val))
      print(f"Epoch {e + 1}: val_acc {val_accuracy:.4f}")


print("SGD")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, torch.optim.SGD(simple_nn.parameters(), lr=3))

print("Adam")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, torch.optim.Adam(simple_nn.parameters(), lr=0.01))

With SGD and the same learning rate, we get the performance of the hand-written loop back: `optimizer.step()` applies the same update rule, and `optimizer.zero_grad()` replaces `model.zero_grad()`. Changing the optimization algorithm now only takes changing the optimizer: Adam adapts each parameter's step, and is used with a much smaller learning rate.

## Moving to Deep Networks

To move to deep networks, we add weight and bias matrices for each layer to our weight creation function, for instance as follows:

In [ ]:
def create_mlp_weights(n_inputs: int,
                       n_outputs: int,
                       hidden_layer_sizes: list[int] = [512],
                      ) -> nn.ModuleList:
  variables = nn.ModuleList()

  # Keep track of the last output size: it will be the new
  # input size. Starts at n_inputs
  last_size = n_inputs

  for i, hidden_layer_size in enumerate(hidden_layer_sizes, 1):
    # Randomly initialize a weight matrix and a bias vector.
    # The weights' variance is 1 / (number of inputs): with a variance of
    # 1, the tanh activations would saturate (see the class on initialization)
    W = torch.randn(last_size, hidden_layer_size, device=device) / last_size ** 0.5
    b = torch.zeros(hidden_layer_size, device=device)

    # Add these two parameters to a dictionary, then to the list of
    # our parameters
    variables.append(nn.ParameterDict(dict(W=nn.Parameter(W),
                                           b=nn.Parameter(b))))

    # Update the last output size used
    last_size = hidden_layer_size

  # The last layer is special: its output size is always n_outputs,
  # so we handle it separately
  W = torch.randn(last_size, n_outputs, device=device) / last_size ** 0.5
  b = torch.zeros(n_outputs, device=device)
  variables.append(nn.ParameterDict(dict(W=nn.Parameter(W),
                                         b=nn.Parameter(b))))
  return variables

As for the model's output, it's now computed iteratively, layer by layer. Each layer takes the result of the previous layer as input. For simplicity, you can choose yourself the activation function you'll use for the hidden layers.

*Complete the `MLP.forward` function to compute the result of a deep network. Each parameter is accessible through `self.weights[layer_number][variable_name]`. For instance, to access the `W` matrix of the first layer: `self.weights[0]["W"]`.*

In [ ]:
class MLP(nn.Module):
  def __init__(self,
               hidden_layer_sizes: list[int] = [512],
               n_inputs: int = 28 ** 2,
               n_outputs: int = 10) -> None:
    super().__init__()
    self.weights = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    # Your code here
    return X

### Solution

In [ ]:
class MLP(nn.Module):
  def __init__(self,
               hidden_layer_sizes: list[int] = [512],
               n_inputs: int = 28 ** 2,
               n_outputs: int = 10) -> None:
    super().__init__()
    self.weights = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    current = X
    for layer_weights in self.weights:
      # Compute the non-activated output of the layer of neurons
      current = current @ layer_weights["W"] + layer_weights["b"]
      # Activation if we're not in the output layer
      if layer_weights is not self.weights[-1]:
        current = torch.tanh(current)
    return torch.log_softmax(current, dim=-1)

## Training the Deep Network

In [ ]:
_ = train(MLP(), learning_rate=0.5, epochs=100)

*What do you observe when training a network with a hidden layer of 512 neurons?*

Your answer here

### Solution

The network with one hidden layer does much better than the one without a hidden layer: about 97% validation accuracy, versus 88%. Training and validation accuracies stay close: no marked overfitting.

This is partly thanks to the weight initialization, with a variance of 1 / (number of inputs). To convince yourself, remove the division by `last_size ** 0.5` in `create_mlp_weights` and rerun: with weights of variance 1, the tanh activations saturate, and the network reaches about 98% on training but only 90% on validation. This large gap looks like overfitting, but it comes from a bad initialization.

## Regularization

Regularization limits overfitting. It becomes useful when the network is large relative to the data, or when it's trained for a long time: here, train for 200 epochs and compare with the previous training.

To set it up, we add a term to the loss function that penalizes large weight values.

*Modify the function below to compute the L2 penalty. It's equal to the sum of the squares of the parameters.*

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def worker(model: nn.Module) -> torch.Tensor:
    weights_norm = torch.zeros((), device=device)
    # Your code here
    return lambda_coefficient * weights_norm

  return worker


train(MLP(),
      epochs=200,
      learning_rate=3e-1,
      regularization=l2(1e-4))

### Solution

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def worker(model: nn.Module) -> torch.Tensor:
    weights_norm = torch.zeros((), device=device)
    for parameter in model.parameters():
      weights_norm = weights_norm + torch.sum(torch.square(parameter))
    return lambda_coefficient * weights_norm

  return worker


train(MLP(),
      epochs=200,
      learning_rate=3e-1,
      regularization=l2(1e-4))  # with 3e-3, the penalty is too strong: underfitting

## Hyperparameter Search

We'll now search for hyperparameters. Use [`random.choice`](https://docs.python.org/3/library/random.html#random.choice) to sample the learning rate and the number of hidden units, and return a list of the parameters and a metric of your choice.

In [ ]:
def search_hyperparameters(
    X_train: torch.Tensor = X_train,
    y_train: torch.Tensor = y_train,
    X_val: torch.Tensor = X_val,
    y_val: torch.Tensor = y_val,
    learning_rates: list[float] = [0.1, 0.5, 1],
    ns_hidden_units: list[int]  = range(32, 513, 32),
    n_iters: int = 5
    ) -> tuple[list[dict[str, float]], list[float]]:
  params = []
  val_accs = []
  for _ in range(n_iters):
    pass
  return params, val_accs


search_hyperparameters()

### Solution

In [ ]:
def search_hyperparameters(
    X_train: torch.Tensor = X_train,
    y_train: torch.Tensor = y_train,
    X_val: torch.Tensor = X_val,
    y_val: torch.Tensor = y_val,
    learning_rates: list[float] = [0.1, 0.5, 1],
    ns_hidden_units: list[int]  = range(32, 513, 32),
    n_iters: int = 5
    ) -> tuple[list[dict[str, float]], list[float]]:
  params = []
  val_accs = []
  for _ in range(n_iters):
    learning_rate = random.choice(learning_rates)
    n_hidden_units = random.choice(ns_hidden_units)
    mlp = MLP(hidden_layer_sizes=[n_hidden_units])
    _, val_acc_epochs, _, _ = train(mlp, X_train, y_train, X_val, y_val,
                                    learning_rate=learning_rate, epochs=50)
    val_accs.append(val_acc_epochs[-1])
    params.append(dict(learning_rate=learning_rate,
                       n_hidden_units=n_hidden_units))
  return params, val_accs


search_hyperparameters()

## Final Evaluation on the Test Set

The test set hasn't been used so far: neither for training nor for choosing hyperparameters. That's what makes its evaluation reliable.

*Train an MLP with the best hyperparameters found, then compute its accuracy on `X_test`. Is it close to the validation accuracy?*

In [ ]:
# Your code here

### Solution

In [ ]:
# Best configuration found (adapt to your results)
mlp = MLP(hidden_layer_sizes=[512])
_ = train(mlp, learning_rate=0.5, epochs=100)
with torch.no_grad():
  test_accuracy = accuracy(y_test, mlp(X_test))
print(f"Test set accuracy: {test_accuracy:.4f}")

The test accuracy (about 96.5%) is close to the validation accuracy (about 96.8%). Since the test set wasn't used for any choice, it's a reliable estimate of the model's performance on new data.